In [1]:
import pandas as pd
import datetime
import os

In [2]:
def get_flag_count(df, col):
    """Safely get flag column count without crashing if column doesn't exist."""
    if col in df.columns:
        return int(df[col].sum())
    return "not run"

def check_non_ascii(series):
    """Count rows with non-ASCII characters in a string column."""
    return series.dropna().apply(
        lambda x: any(ord(c) > 127 for c in str(x))
    ).sum()

In [6]:
def validate_and_clean_orders(filepath, report_dir='../reports', date_cutoff=None):
    
    report = []
    report.append("=" * 60)
    report.append("DATA VALIDATION & CLEANING REPORT")
    report.append(f"Table: orders")
    report.append(f"Run timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append(f"Source file: {filepath}")
    report.append("=" * 60)

    # ── PHASE 0: STRUCTURAL CHECK ──────────────────────────────
    report.append("\n[ PHASE 0 - STRUCTURAL CHECK ]")
    
    orders = pd.read_csv(filepath)
    original_row_count = len(orders)
    
    expected_columns = [
        'order_id', 'customer_id', 'order_status',
        'order_purchase_timestamp', 'order_approved_at',
        'order_delivered_carrier_date', 'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
    
    missing_cols = [c for c in expected_columns if c not in orders.columns]
    extra_cols = [c for c in orders.columns if c not in expected_columns]
    
    report.append(f"Row count: {original_row_count}")
    report.append(f"Column count: {len(orders.columns)}")
    report.append(f"Missing expected columns: {missing_cols if missing_cols else 'None'}")
    report.append(f"Unexpected extra columns: {extra_cols if extra_cols else 'None'}")
    
    if missing_cols:
        print(f"⚠ WARNING: Missing columns detected: {missing_cols}")

    # ── PHASE 1: DUPLICATES ────────────────────────────────────
    report.append("\n[ PHASE 1 - DUPLICATES ]")
    
    full_dupes = orders.duplicated().sum()
    orders = orders.drop_duplicates()
    report.append(f"Full-row duplicates removed: {full_dupes}")
    
    dupe_key_mask = orders['order_id'].duplicated(keep=False)
    dupe_keys = dupe_key_mask.sum()
    report.append(f"Duplicate order_id with conflicting data (flagged, kept): {dupe_keys}")
    orders['flag_duplicate_order_id'] = dupe_key_mask

    # ── PHASE 2: KEY UNIQUENESS ────────────────────────────────
    report.append("\n[ PHASE 2 - KEY UNIQUENESS ]")
    report.append(f"order_id is unique: {orders['order_id'].is_unique}")

    # ── PHASE 3: TYPE CONVERSION & STANDARDIZATION ─────────────
    report.append("\n[ PHASE 3 - TYPE CONVERSION & STANDARDIZATION ]")
    
    date_cols = {
        'order_purchase_timestamp': 'purchase_ts',
        'order_approved_at': 'approved_ts',
        'order_delivered_carrier_date': 'carrier_delivered_dt',
        'order_delivered_customer_date': 'customer_delivered_dt',
        'order_estimated_delivery_date': 'est_delivery_dt'
    }
    
    for original, renamed in date_cols.items():
        orders = orders.rename(columns={original: renamed})
        orders[renamed] = pd.to_datetime(orders[renamed], errors='coerce')
    
    report.append(f"Date columns renamed and converted to datetime: {list(date_cols.values())}")
    # ── PHASE 4: CATEGORICAL COLUMNS ──────────────────────────
    report.append("\n[ PHASE 4 - CATEGORICAL COLUMNS ]")
    
    known_statuses = ['delivered', 'shipped', 'canceled', 'unavailable',
                      'invoiced', 'processing', 'created', 'approved']
    
    # normalize casing and whitespace first
    orders['order_status'] = orders['order_status'].str.strip().str.lower()
    
    unexpected_status = [
        v for v in orders['order_status'].unique() 
        if v not in known_statuses
    ]
    
    orders['flag_unexpected_status'] = orders['order_status'].apply(
        lambda x: x not in known_statuses if pd.notna(x) else False
    )
    
    report.append(f"order_status - unique values: {orders['order_status'].nunique()}")
    report.append(f"order_status - unexpected values: {unexpected_status if unexpected_status else 'None'}")
    report.append(f"order_status - null count: {orders['order_status'].isnull().sum()}")

    # ── PHASE 5: NUMERICAL (skipped for orders — lives in order_items) ──
    report.append("\n[ PHASE 5 - NUMERICAL COLUMNS ]")
    report.append("No numerical columns in orders table — handled in order_items pipeline")

    # ── PHASE 6: DATE VALIDATION ───────────────────────────────
    report.append("\n[ PHASE 6 - DATE VALIDATION ]")
    
    # null counts per date column
    date_renamed = ['purchase_ts', 'approved_ts', 'carrier_delivered_dt', 
                    'customer_delivered_dt', 'est_delivery_dt']
    for col in date_renamed:
        report.append(f"{col} - null count: {orders[col].isnull().sum()}")
    
    # future date check using cutoff
    cutoff = pd.Timestamp(date_cutoff) if date_cutoff else pd.Timestamp.now()
    for col in ['purchase_ts', 'approved_ts', 'carrier_delivered_dt', 'customer_delivered_dt']:
        future = orders[orders[col] > cutoff]
        report.append(f"{col} - dates beyond cutoff ({cutoff.date()}): {len(future)}")

    # cross-column sequence checks
    orders['flag_carrier_before_approval'] = (
        orders['carrier_delivered_dt'] < orders['approved_ts']
    )
    orders['flag_carrier_approval_large_gap'] = (
        (orders['carrier_delivered_dt'] < orders['approved_ts']) &
        ((orders['approved_ts'] - orders['carrier_delivered_dt']) > pd.Timedelta(days=7))
    )
    orders['flag_delivery_sequence'] = (
        orders['customer_delivered_dt'] < orders['carrier_delivered_dt']
    )
    orders['flag_delivered_no_date'] = (
        (orders['order_status'] == 'delivered') &
        (orders['customer_delivered_dt'].isnull())
    )
    orders['flag_delivered_missing_carrier_too'] = (
        (orders['order_status'] == 'delivered') &
        (orders['customer_delivered_dt'].isnull()) &
        (orders['carrier_delivered_dt'].isnull())
    )

    report.append(f"Approved before purchase: {get_flag_count(orders, 'flag_approved_before_purchase')}")
    report.append(f"Carrier before approval (all): {get_flag_count(orders, 'flag_carrier_before_approval')}")
    report.append(f"Carrier before approval (>7 days): {get_flag_count(orders, 'flag_carrier_approval_large_gap')}")
    report.append(f"Customer delivered before carrier: {get_flag_count(orders, 'flag_delivery_sequence')}")
    report.append(f"Delivered status, missing customer date: {get_flag_count(orders, 'flag_delivered_no_date')}")
    report.append(f"Delivered status, missing both dates: {get_flag_count(orders, 'flag_delivered_missing_carrier_too')}")

    # ── PHASE 7: TEXT & SPECIAL CHARACTERS ────────────────────
    report.append("\n[ PHASE 7 - TEXT & SPECIAL CHARACTERS ]")
    
    id_cols = ['order_id', 'customer_id']
    for col in id_cols:
        length_counts = orders[col].str.len().value_counts()
        consistent = len(length_counts) == 1
        report.append(f"{col} - length consistent: {consistent} {dict(length_counts) if not consistent else '(all 32 chars)'}")
        non_ascii = check_non_ascii(orders[col])
        report.append(f"{col} - non-ASCII characters: {non_ascii}")
    
    report.append("PII exposure: None - IDs are anonymized hashes")

    # ── PHASE 8: SUMMARY REPORT ────────────────────────────────
    report.append("\n[ PHASE 8 - SUMMARY ]")
    
    flag_cols = [col for col in orders.columns if col.startswith('flag_')]
    total_flagged_rows = orders[flag_cols].any(axis=1).sum() if flag_cols else 0
    
    report.append(f"Rows in (original): {original_row_count}")
    report.append(f"Rows out (after cleaning): {len(orders)}")
    report.append(f"Rows auto-fixed (duplicates removed): {original_row_count - len(orders)}")
    report.append(f"Total rows with at least one flag: {total_flagged_rows}")
    report.append(f"Flag columns created: {flag_cols}")
    report.append("=" * 60)

    # save report
    os.makedirs(report_dir, exist_ok=True)
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    report_path = f"{report_dir}/orders_validation_{timestamp}.txt"
    
    with open(report_path, 'w') as f:
        f.write("\n".join(report))
    
    print("\n".join(report))
    print(f"\n✅ Report saved to: {report_path}")
    
    return orders

In [7]:
orders_clean = validate_and_clean_orders(
    '../csv-files/olist_orders_dataset.csv',
    date_cutoff='2018-09-03'
)

DATA VALIDATION & CLEANING REPORT
Table: orders
Run timestamp: 2026-06-22 15:36:30
Source file: ../csv-files/olist_orders_dataset.csv

[ PHASE 0 - STRUCTURAL CHECK ]
Row count: 99441
Column count: 8
Missing expected columns: None
Unexpected extra columns: None

[ PHASE 1 - DUPLICATES ]
Full-row duplicates removed: 0
Duplicate order_id with conflicting data (flagged, kept): 0

[ PHASE 2 - KEY UNIQUENESS ]
order_id is unique: True

[ PHASE 3 - TYPE CONVERSION & STANDARDIZATION ]
Date columns renamed and converted to datetime: ['purchase_ts', 'approved_ts', 'carrier_delivered_dt', 'customer_delivered_dt', 'est_delivery_dt']

[ PHASE 4 - CATEGORICAL COLUMNS ]
order_status - unique values: 8
order_status - unexpected values: None
order_status - null count: 0

[ PHASE 5 - NUMERICAL COLUMNS ]
No numerical columns in orders table — handled in order_items pipeline

[ PHASE 6 - DATE VALIDATION ]
purchase_ts - null count: 0
approved_ts - null count: 160
carrier_delivered_dt - null count: 1783
cus